In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/base.csv", parse_dates=["date", "date_of_birth"])
df = df[df["date"] >= "2013-07-03"].reset_index(drop=True)
df["row_id"] = df.index  # unique id for each valuation row

app = pd.read_csv(
    "../data/raw/appearances.csv",
    usecols=["player_id", "date", "goals", "assists", "minutes_played", "yellow_cards", "red_cards"],
    parse_dates=["date"],
)
print(df.shape, app.shape)

(553724, 15) (1894350, 7)


In [2]:
app = app.sort_values(["player_id", "date"]).reset_index(drop=True)
stat_cols = ["goals", "assists", "minutes_played", "yellow_cards", "red_cards"]

# Cumulative stats per player, with a row for each game
cum = app.copy()
for c in stat_cols:
    cum[c] = cum.groupby("player_id")[c].cumsum()
cum["games"] = cum.groupby("player_id").cumcount() + 1

def cum_at(target_dates):
    """Cumulative stats for each valuation row at (date - 1 day), strictly before the valuation."""
    left = df[["row_id", "player_id"]].copy()
    left["lookup_date"] = target_dates
    left = left.sort_values("lookup_date")
    out = pd.merge_asof(
        left,
        cum.sort_values("date"),
        left_on="lookup_date", right_on="date",
        by="player_id", direction="backward",
    )
    return out.set_index("row_id")[stat_cols + ["games"]].fillna(0)

now = cum_at(df["date"] - pd.Timedelta(days=1))
past = cum_at(df["date"] - pd.Timedelta(days=366))

feat = (now - past).add_suffix("_12m")
df = df.join(feat, on="row_id")

# Per-90 rates (guard against tiny minutes)
mins = df["minutes_played_12m"].clip(lower=1)
df["goals_per90"] = np.where(df["minutes_played_12m"] >= 450, df["goals_12m"] / mins * 90, np.nan)
df["assists_per90"] = np.where(df["minutes_played_12m"] >= 450, df["assists_12m"] / mins * 90, np.nan)

print(df[["age", "games_12m", "goals_12m", "minutes_played_12m", "goals_per90"]].describe())

                 age      games_12m      goals_12m  minutes_played_12m  \
count  553724.000000  553724.000000  553724.000000       553724.000000   
mean       25.487590       7.530461       0.753048          516.818023   
std         4.613843      12.292621       2.459370          915.655706   
min        15.000684       0.000000       0.000000            0.000000   
25%        21.828884       0.000000       0.000000            0.000000   
50%        24.950034       0.000000       0.000000            0.000000   
75%        28.689938      12.000000       0.000000          695.000000   
max        41.998631      68.000000      61.000000         5790.000000   

         goals_per90  
count  157985.000000  
mean        0.128632  
std         0.170820  
min         0.000000  
25%         0.000000  
50%         0.065550  
75%         0.188976  
max         2.232558  


In [3]:
print("Rows with zero games in last 12m:", (df["games_12m"] == 0).mean().round(3))
print("Negative values (should be 0):", (df[[c for c in df.columns if c.endswith("_12m")]] < 0).sum().sum())
df[["player_id", "date", "market_value_in_eur", "games_12m", "goals_12m", "minutes_played_12m"]].head(10)

Rows with zero games in last 12m: 0.602
Negative values (should be 0): 0


,player_id,date,market_value_in_eur,games_12m,goals_12m,minutes_played_12m
0,274811,2013-07-03,50000,0.0,0.0,0.0
1,3391,2013-07-05,400000,0.0,0.0,0.0
2,6076,2013-07-05,100000,13.0,2.0,692.0
3,6717,2013-07-05,150000,12.0,0.0,870.0
4,7101,2013-07-05,700000,0.0,0.0,0.0
5,7103,2013-07-05,200000,0.0,0.0,0.0
6,7311,2013-07-05,150000,0.0,0.0,0.0
7,8539,2013-07-05,200000,0.0,0.0,0.0
8,14195,2013-07-05,400000,0.0,0.0,0.0
9,16768,2013-07-05,100000,0.0,0.0,0.0


In [4]:
tmp = df.assign(zero=(df["games_12m"] == 0))
by_league = (
    tmp.groupby("player_club_domestic_competition_id")
    .agg(rows=("zero", "size"), zero_share=("zero", "mean"))
    .query("rows >= 2000")
    .sort_values("zero_share")
)
print(by_league.round(2).to_string())

                                      rows  zero_share
player_club_domestic_competition_id                   
GB1                                  27226        0.36
BE1                                  22768        0.40
FR1                                  25864        0.40
SC1                                  14029        0.40
NL1                                  25076        0.41
UKR1                                 18283        0.42
DK1                                  16224        0.42
PO1                                  25615        0.43
ES1                                  30251        0.43
GR1                                  24031        0.45
TR1                                  36932        0.45
L1                                   29030        0.47
RU1                                  26081        0.50
IT1                                  40563        0.56
SER1                                  4839        0.83
KR1                                   3682        0.85
TS1       

In [5]:
df["has_stats"] = (df["games_12m"] > 0).astype(int)

perf_cols = [c for c in df.columns if c.endswith("_12m")]
df.loc[df["has_stats"] == 0, perf_cols] = np.nan

print(df["has_stats"].mean().round(3))
df.to_csv("../data/processed/features_v1.csv", index=False)

0.398


In [6]:
# Leagues where appearances data exists (clear gap between 0.56 and 0.83)
COVERED = by_league[by_league["zero_share"] < 0.6].index.tolist()
print(len(COVERED), COVERED)

df["league_covered"] = df["player_club_domestic_competition_id"].isin(COVERED).astype(int)

count_cols = ["games_12m", "goals_12m", "assists_12m",
              "minutes_played_12m", "yellow_cards_12m", "red_cards_12m"]
mask = df["league_covered"] == 1
df.loc[mask, count_cols] = df.loc[mask, count_cols].fillna(0)

df.to_csv("../data/processed/features_v1.csv", index=False)

14 ['GB1', 'BE1', 'FR1', 'SC1', 'NL1', 'UKR1', 'DK1', 'PO1', 'ES1', 'GR1', 'TR1', 'L1', 'RU1', 'IT1']


In [7]:
cov = df[df["league_covered"] == 1]
buckets = pd.cut(
    cov["market_value_in_eur"],
    [0, 2.5e5, 1e6, 5e6, 2e7, 1e9],
    labels=["<250k", "250k-1M", "1-5M", "5-20M", "20M+"],
)
print(cov.groupby(buckets, observed=True)["has_stats"].apply(lambda s: 1 - s.mean()).round(2))

market_value_in_eur
<250k      0.78
250k-1M    0.45
1-5M       0.18
5-20M      0.03
20M+       0.00
Name: has_stats, dtype: float64


In [8]:
df.to_csv("../data/processed/features_v1.csv", index=False)

In [9]:
# Sorting by date so every group's rows are already in time order
df = df.sort_values("date").reset_index(drop=True)
df["lv"] = np.log1p(df["market_value_in_eur"])

dates = df["date"].to_numpy()
lv = df["lv"].to_numpy()
ONE_YEAR = np.timedelta64(365, "D")

def window_sums(keys):
    """For each row, sum and count of log-values in the 365 days strictly before its date, within its group."""
    sums = np.zeros(len(df))
    cnts = np.zeros(len(df))
    for _, idx in df.groupby(keys, sort=False).indices.items():
        d, v = dates[idx], lv[idx]                      # rows of one group, already date-ordered
        csum = np.concatenate([[0], np.cumsum(v)])      # running total, with a leading 0
        hi = np.searchsorted(d, d, side="left")         # first row at or after this date (excludes today and the future)
        lo = np.searchsorted(d, d - ONE_YEAR, side="left")  # first row inside the 365-day window
        sums[idx] = csum[hi] - csum[lo]
        cnts[idx] = hi - lo
    return sums, cnts

def add_context(prefix, keys, min_n):
    gs, gc = window_sums(keys)                          # everyone in the club/league
    ps, pc = window_sums(keys + ["player_id"])          # the player's own earlier rows there
    n = gc - pc                                         # subtracting the player so the target never feeds its own feature
    df[f"{prefix}_n_prev"] = n
    safe_n = np.where(n > 0, n, 1)                      # avoiding division by zero
    df[f"{prefix}_avg_lv_prev"] = np.where(n >= min_n, (gs - ps) / safe_n, np.nan)

add_context("club", ["current_club_id"], min_n=5)
add_context("league", ["player_club_domestic_competition_id"], min_n=50)

df["age_sq"] = df["age"] ** 2

print(df[["club_n_prev", "club_avg_lv_prev", "league_avg_lv_prev"]].describe())
print("Club feature NaN share:", df["club_avg_lv_prev"].isna().mean().round(3))
print("Correlation with log value:")
print(df[["club_avg_lv_prev", "league_avg_lv_prev", "age", "minutes_played_12m", "lv"]].corr()["lv"].round(2))

         club_n_prev  club_avg_lv_prev  league_avg_lv_prev
count  553724.000000     499164.000000       550478.000000
mean       57.329919         13.314906           13.200796
std        42.913123          0.956341            0.738105
min         0.000000          9.784085           11.411560
25%        20.000000         12.648008           12.618448
50%        53.000000         13.151721           13.082485
75%        86.000000         13.828000           13.659758
max       291.000000         17.760577           16.195398
Club feature NaN share: 0.099
Correlation with log value:
club_avg_lv_prev      0.57
league_avg_lv_prev    0.46
age                   0.13
minutes_played_12m    0.65
lv                    1.00
Name: lv, dtype: float64


In [10]:
df = df.drop(columns=["lv"])
df.to_csv("../data/processed/features_v2.csv", index=False)
print(df.shape)

(553724, 30)
